# ProCIR training on a Jupyter GPU server

Run the cells top to bottom. Every step can be re-run: the repo is pulled, finished downloads and
extracted zips are skipped, and a stopped training run resumes from its last checkpoint.

1. **Parameters**: paste the Google Drive folder link (shared as *Anyone with the link*) holding the
   `procir_*.zip` files made by `scripts/pack_data.py`.
2. **Setup**: clone the repo, install packages, download and extract the data, check the GPU.
3. **Pilot** (about 15 min): 20 training steps to measure seconds per step and memory, then pick
   `train_fraction` so each run trains in about `TARGET_TRAIN_HOURS`.
4. **Launch**: all runs in `RUNS` go into one background queue (`nohup`): train → export → eval with the
   unmodified upstream `evaluate.py`. It keeps running if the browser or kernel disconnects.
5. **Monitor** and **Results**: re-run these cells any time.

In [ ]:
# ---- parameters ----
import os
REPO_URL = "https://github.com/vbphuong18/HCMUT_Fashion_MV.git"
REPO_DIR = os.path.expanduser("~/HCMUT_Fashion_MV")  # home needs ~45 GB: create the notebook with a 100 Gi workspace volume
GDRIVE_FOLDER = "https://drive.google.com/drive/folders/PASTE_FOLDER_ID_HERE"
CONFIG = "configs/procir_fast.yaml"     # MT+Align, all three sources, 3 views at 336^2, LoRA, B=64
TARGET_TRAIN_HOURS = 7.0                # training time per run; eval adds roughly 1-2 h per run
MAX_TRAIN_FRACTION = 0.25               # the proposal's upper bound
CHUNK_SIZE = 8                          # GradCache chunk; the pilot cell lowers it if memory is short
RUNS = [                                # (output name, extra config overrides); run in this order
    ("fast_mt_align_s42", []),
    ("fast_single_s42", ["multi_turn=false", "align=false"]),   # single-turn baseline (paper Table 3, row 1)
]
EVAL_PROTOCOLS = ["upstream", "train"]  # upstream = 5 views / 512^2 (reported); train = 3 views / 336^2
EXPORT_CHECKPOINT = "best"             # best = highest dev mean R@5 (ckpt/best.pt); "latest" = final step
RUN_GPU_TESTS = True

In [ ]:
import json
import shlex
import subprocess
import sys
import time
from pathlib import Path


def sh(cmd, check=True, cwd=None, env=None):
    """Run a shell command and stream its output into the notebook."""
    print("+", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or (REPO_DIR if Path(REPO_DIR).exists() else None),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() and check:
        raise RuntimeError(f"exit code {p.returncode}: {cmd}")
    return p.returncode


PY = shlex.quote(sys.executable)
ENV = {"PYTHONPATH": "src:external/FashionMV", "TOKENIZERS_PARALLELISM": "false", "PYTHONUNBUFFERED": "1"}

## 1. Setup

In [ ]:
# ---- code ----
if Path(REPO_DIR, ".git").exists():
    sh("git pull --ff-only && git submodule update --init")
else:
    sh(f"git clone --recurse-submodules {REPO_URL} {shlex.quote(REPO_DIR)}", cwd=os.path.expanduser("~"))
os.chdir(REPO_DIR)
sh("git log --oneline -1 && git -C external/FashionMV rev-parse --short HEAD")

In [ ]:
# ---- packages; the linear-attention kernels one at a time so one failed build cannot block the rest ----
sh(f"{PY} -m pip install -q -r requirements-train.txt gdown")
sh(f"{PY} -m pip install -q flash-linear-attention", check=False)
has_nvcc = sh("nvcc --version", check=False) == 0
if has_nvcc:
    sh(f"MAX_JOBS=8 {PY} -m pip install -q causal-conv1d --no-build-isolation", check=False)
else:
    print("no nvcc: causal-conv1d stays on its PyTorch fallback (slower convolution only)")

In [ ]:
# ---- data: download the zips from Drive, extract into data/, check every triplet has images ----
# one zip at a time (download, extract, delete): needs ~19 GB for the data plus ~4 GB for one zip
sh("df -h . ~", check=False)
sh(f"{PY} scripts/fetch_data.py --gdrive-folder {shlex.quote(GDRIVE_FOLDER)}")

In [ ]:
# ---- GPU check: strict first, then accept missing kernels (slower) ----
ALLOW_SLOW = sh(f"{PY} scripts/check_env.py runs/env_check.json", check=False) != 0
if ALLOW_SLOW:
    sh(f"{PY} scripts/check_env.py --allow-slow runs/env_check.json")
import torch
free, total = torch.cuda.mem_get_info()
print(f"GPU memory free {free / 2**30:.1f} / {total / 2**30:.1f} GiB")
if free < 18 * 2**30:
    print("WARNING: less than 18 GiB free (another process holds memory); using CHUNK_SIZE = 4")
    CHUNK_SIZE = min(CHUNK_SIZE, 4)

In [ ]:
# ---- GPU tests (about 10 min): export matches upstream evaluate.py, padding invariance, resume ----
if RUN_GPU_TESTS:
    sh(f"{PY} -m pytest -q -m gpu -x", env=ENV)

## 2. Pilot: seconds per step → train_fraction

In [ ]:
def pipeline_cmd(name, extra, stages=("train", "export", "eval")):
    sets = [f"output_dir=runs/{name}", f"chunk_size={CHUNK_SIZE}", *extra]
    flags = " --allow-slow" if ALLOW_SLOW else ""
    return (f"{PY} -m procir_train.pipeline --config {CONFIG} --set {' '.join(sets)} "
            f"--stages {' '.join(stages)} --eval-protocols {' '.join(EVAL_PROTOCOLS)} "
            f"--export-checkpoint {EXPORT_CHECKPOINT}{flags}")


def pilot(chunk):
    sh("rm -rf runs/pilot")
    global CHUNK_SIZE
    CHUNK_SIZE = chunk
    cmd = pipeline_cmd("pilot", ["stop_after_steps=20", "log_every=1", "eval_every=0", "save_every=100000"],
                       stages=("train",))
    return sh(cmd, check=False, env=ENV) == 0


for chunk in sorted({c for c in (CHUNK_SIZE, 4, 2) if c <= CHUNK_SIZE}, reverse=True):
    if pilot(chunk):
        break
    print(f"pilot failed with chunk_size={chunk} (probably out of memory); trying a smaller chunk")
else:
    raise RuntimeError("pilot failed even with chunk_size=2; see the log above")

sh(f"{PY} scripts/summarize_run.py runs/pilot runs/pilot_summary.json")
summary = json.loads(Path("runs/pilot_summary.json").read_text())
sec = summary["sec_per_step_median"]
stats = json.loads(Path("runs/pilot/data_stats.json").read_text())
pool = sum(v["train_pool"] for k, v in stats.items() if isinstance(v, dict) and "train_pool" in v)
steps = int(TARGET_TRAIN_HOURS * 3600 / sec)
TRAIN_FRACTION = round(min(MAX_TRAIN_FRACTION, steps * 64 / pool), 3)
print(f"chunk_size={CHUNK_SIZE}  sec/step={sec:.1f}  peak reserved={summary.get('max_mem_reserved_gb')} GB")
print(f"train pool {pool} triplets -> train_fraction={TRAIN_FRACTION} "
      f"(~{int(TRAIN_FRACTION * pool / 64)} steps, ~{TRAIN_FRACTION * pool / 64 * sec / 3600:.1f} h per run)")

## 3. Launch all runs in a background queue

In [ ]:
Path("runs").mkdir(exist_ok=True)
lines = ["#!/usr/bin/env bash", f"cd {shlex.quote(REPO_DIR)}",
         "export PYTHONPATH=src:external/FashionMV TOKENIZERS_PARALLELISM=false PYTHONUNBUFFERED=1"]
for name, extra in RUNS:
    lines.append(f"echo \"=== {name} start $(date)\"")
    lines.append(pipeline_cmd(name, [f"train_fraction={TRAIN_FRACTION}", *extra]) + f" > runs/{name}.log 2>&1")
    lines.append(f"echo \"=== {name} exit $? $(date)\"")
Path("runs/queue.sh").write_text("\n".join(lines) + "\n")
print(Path("runs/queue.sh").read_text())
sh("nohup bash runs/queue.sh > runs/queue.log 2>&1 &")

## 4. Monitor (re-run any time)

In [ ]:
sh("cat runs/queue.log; nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv", check=False)
for name, _ in RUNS:
    run = Path("runs", name)
    print(f"\n## {name}:", (run / "STATUS").read_text().strip() if (run / "STATUS").exists() else "")
    log = run / "train_log.jsonl"
    if log.exists():
        rows = [json.loads(l) for l in log.read_text().splitlines() if l.strip()]
        for r in [r for r in rows if "loss" in r][-3:]:
            print({k: round(v, 4) if isinstance(v, float) else v for k, v in r.items()
                   if k in ("step", "loss", "l_cir", "l_doc", "l_src", "lr", "grad_norm", "sec_per_step", "max_mem_reserved_gb")})
        for r in [r for r in rows if "dev" in r][-1:]:
            print("dev:", r)
    if Path("runs", f"{name}.log").exists():
        sh(f"tail -n 3 runs/{name}.log", check=False)

## 5. Results

In [ ]:
table = []
for name, _ in RUNS:
    up = Path("runs", name, "eval_upstream", "eval_results.json")
    ours = Path("runs", name, "eval_upstream.json")  # single-turn runs are scored by retrieval_eval
    res = json.loads(up.read_text()) if up.exists() else (json.loads(ours.read_text()) if ours.exists() else None)
    tr = Path("runs", name, "eval_train.json")
    table.append((name, res, json.loads(tr.read_text()) if tr.exists() else None))
for name, res, tr in table:
    print(f"\n### {name}")
    print("upstream protocol (5 views, 512^2):", json.dumps(res, indent=1) if res else "not yet")
    print("train protocol (3 views, 336^2):", json.dumps(tr, indent=1) if tr else "not yet")

In [ ]:
# ---- bundle small result files to download (no checkpoints) ----
sh("tar -czf runs/results_bundle.tgz runs/*.log runs/env_check.json runs/pilot_summary.json "
   "runs/*/config.yaml runs/*/train_log.jsonl runs/*/data_stats.json runs/*/env.json runs/*/STATUS "
   "runs/*/eval_upstream runs/*/eval_*.json 2>/dev/null; ls -la runs/results_bundle.tgz", check=False)